# 01 Token 分块 + RAG 工具集成
基于 agent-advanced 框架，实现 token 级别的文档分块，并将 RAG 能力作为 Agent 工具集成。

In [ ]:
import sys
sys.path.insert(0, '..')

from src.capabilities.token_chunker import TokenChunker, load_markdown_files
from src.agent_framework.embedding_store import EmbeddingStore
from src.agent_framework.llm import LLMClient
from src.agent_framework.core import Agent
from src.capabilities.demo_tools import create_demo_tools

## Token Chunker 演示

In [ ]:
sample_md = """# Transformer 架构

Transformer 是由 Google 在 2017 年提出的深度学习架构。

## Self-Attention 机制

Self-Attention 是 Transformer 的核心组件。它允许模型在处理序列时，
关注序列中所有位置的信息，而不是像 RNN 那样只能看到过去的信息。

具体来说，对于输入序列中的每个元素，Self-Attention 计算该元素与所有其他元素的
注意力权重，然后对所有元素的值进行加权求和。

## 多头注意力

Multi-Head Attention 将 Self-Attention 并行执行多次，
每次使用不同的线性投影矩阵，从而让模型从不同的表示子空间中学习信息。

## 位置编码

由于 Transformer 不像 RNN 那样天然具有顺序信息，
需要额外添加位置编码 (Positional Encoding) 来让模型感知序列中元素的位置。
"""

chunker = TokenChunker(chunk_tokens=128, overlap_tokens=32)
print(f"总 token 数: {chunker.count_tokens(sample_md)}")

chunks = chunker.chunk(sample_md, "transformer.md")
for c in chunks:
    m = c["meta"]
    print(f"\nChunk {m['chunk_index']} (tokens {m['token_start']}-{m['token_end']}):")
    print(c["text"][:120] + "..." if len(c["text"]) > 120 else c["text"])

## EmbeddingStore 索引 + 检索

In [ ]:
store = EmbeddingStore()

for c in chunks:
    store.add("documents", c["text"], c["meta"])

print(f"documents collection 大小: {store.collection_size('documents')}")

In [ ]:
queries = [
    "什么是 Self-Attention？",
    "多头注意力的作用是什么？",
    "为什么要加位置编码？",
    "今天天气怎么样？",  # 不相关查询，测试阈值过滤
]

for q in queries:
    print(f"\n🔍 {q}")
    results = store.search("documents", q, top_k=3, threshold=0.3)
    if not results:
        print("  → 无相关结果（阈值过滤）")
    for r in results:
        print(f"  [{r['meta']['chunk_index']}] ({r['score']:.3f}) {r['text'][:80]}...")

## 集成到 Agent

In [ ]:
tools = create_demo_tools(es=store)

agent = Agent(
    tools=tools,
    system_prompt=(
        "你是一个基于知识库回答问题的助手。"
        "回答用户问题前，先调用 search_docs 检索相关文档，"
        "然后根据检索结果回答。如果文档中没有相关信息，就说不知道。"
    ),
    max_rounds=10,
)

In [ ]:
response = agent.chat("Self-Attention 是怎么工作的？")

In [ ]:
response = agent.chat("Transformer 为什么要用位置编码？")

In [ ]:
response = agent.chat("今天天气怎么样？")

## 使用 index_documents 工具索引真实文件

In [ ]:
agent.reset()
response = agent.chat("请先索引 ../data/ 目录下的所有 markdown 文档")